# 01 — Análise exploratória

**CRISP-DM fase 2.3 — exploração dos dados** · RH Analytics

---

Este notebook percorre o **roteiro de análise** da análise exploratória sobre a
base de RH, usando o dicionário de dados montado em
[`00-dicionario-dados.ipynb`](00-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


# Análise exploratória

Nessa fase será realizado uma análise exploratória dos dados seguindo as etapas apresentadas a seguir:

<img src="../docs/imagens/eda-steps.png">

1. Na distribuição dos atributos é possível identificar o comportamento de cada uma das características analisadas
2. A análise estatística univariada analisa a distribuição e dispersão dos dados: a análise da distribuição de frequências e a análise das medidas de localização, dispersão, assimetria e curtose dos dados. 
3.  Na avaliação estatística da causalidade, convencionou-se descrever essas análises bivariadas como formas de investigar se a variação de uma característica pode ser explicada pela variação de outra medida. 
4.  Análise multivariada é uma ferramenta que encontra padrões e relações entre várias variáveis, permitindo prever efeitos e mudanças que uma variável terá sobre a outra. 
5.  Detecção de valores ausentes é uma tarefa comum na área de ciência de dados para mensurar os impactos dessa ausência no conjunto de dados em estudo. 
6.  A detecção de outliers consiste na identificação de padrões em dados com um comportamento diferente do esperado. Estes padrões são muitas vezes referidos como anomalias. 
7.  Em probabilidade e estatística, correlação, dependência ou associação é qualquer relação estatística entre duas ou mais variáveis. 
8.  Os métodos de seleção de características têm a missão de encontrar uma combinação adequada de características relevantes para o aprendizado de um conceito.

# Importações

In [ ]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# Importações
# import prince
import textwrap
# import circlify
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection

: 

In [ ]:
# Funções para aplicar as escalas
def carregar_escalas(caminho):
    escala = pd.read_csv(caminho,
                         sep=None, engine='python',   # detecta , ou ;
                         dtype={'coluna': str, 'nivel': str},
                         encoding='utf-8-sig')        # remove BOM, se houver
    escala.columns = escala.columns.str.strip().str.lower()

    esperadas = {'coluna', 'nivel', 'ordem'}
    faltando = esperadas - set(escala.columns)
    if faltando:
        raise ValueError(f'Colunas ausentes em {caminho}: {faltando}. '
                         f'Colunas encontradas: {escala.columns.tolist()}')

    escala = escala.sort_values(['coluna', 'ordem'])
    return escala.groupby('coluna')['nivel'].apply(list).to_dict()


def aplicar_escalas(df, escalas):
    for coluna, niveis in escalas.items():
        if coluna not in df.columns:
            print(f'Aviso: a coluna "{coluna}" está na escala, mas não existe no dataset')
            continue

        # Se a coluna for numérica (ex.: notas 1 a 5), converte os níveis para número
        if pd.api.types.is_numeric_dtype(df[coluna]):
            niveis = pd.to_numeric(pd.Series(niveis)).tolist()

        df[coluna] = pd.Categorical(df[coluna], categories=niveis, ordered=True)
    return df

In [ ]:
# Carregando base de dados
df = pd.read_csv('../data/raw/base_rh_ptBR.csv') # Ajuste o caminho se sua base estiver em outra pasta

# Carregando dicionário dos dados
dicionario = pd.read_csv('../data/processed/dicionario.csv', sep=';')

# Carregando escalas
escalas = carregar_escalas(caminho='../data/processed/escala.csv')

# DataFrame de escalas (para mapeamento coluna/nivel/ordem)
escalas_df = pd.read_csv('../data/processed/escala.csv', sep=';', dtype={'coluna': str, 'nivel': str})

# Aplicando escalas
df = aplicar_escalas(df, escalas)
# Pasta para tabelas exportadas (rastreabilidade)
from pathlib import Path
PASTA_TABELAS = Path('../docs/tabelas')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)


In [ ]:
# selecionando as mesmas colunas do dicionário de dados
df = df.loc[:,dicionario.variavel]

In [ ]:
dicionario

# Análises iniciais

In [ ]:
# Quantidade de registros
df.info()

In [ ]:
# Quantidade por tipo
dicionario.Tipo.value_counts()

In [ ]:
# Estatísticas descritivas
df.describe()

# Análise univariada

## Características nominais

In [ ]:
#Obter uma lista das variáveis nominais
nominal_columns = (
    dicionario
    .query('Tipo == "Nominal"')
    .variavel
    .to_list()
)

In [ ]:
if nominal_columns != []:
    for column in df.columns:
        if (column in nominal_columns):
            # Tabela de frequência
            freq = df[column].value_counts().reset_index()
            freq.columns = ['categoria', 'frequencia']
            freq['percentual'] = (freq['frequencia'] / len(df) * 100).round(2)
            freq = freq.sort_values('frequencia', ascending=False).reset_index(drop=True)

            # Salvar tabela
            PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
            freq.to_csv(PASTA_TABELAS / f'univariada-nominal-{column}.csv', index=False, encoding='utf-8', sep=';')
            print(f"Tabela salva: univariada-nominal-{column}.csv")

            # Criar figura com as top 10 categorias
            fig, ax = plt.subplots(figsize=(25, 8))
            sns.set_style("whitegrid")
            top10 = freq.head(10)

            sns.barplot(data=top10, x='frequencia', y='categoria', color='#183EFF', alpha=1, ax=ax)

            for bars in ax.containers:
                ax.bar_label(bars, label_type='edge', labels=[f"{v:.0f}\n{p:.1f}%" for v, p in zip(top10['frequencia'], top10['percentual'])])

            ax.axvline(freq['frequencia'].mean(), color="#FFD000", clip_on=False, ls='dashed')

            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel('Quantidade de registros')
            plt.title('As categorias mais frequentes da característica ' + column)

            fig.savefig('../docs/imagens/eda/univariada-nominal-'+column+'.png', bbox_inches='tight')
            plt.close(fig)


## Características contínuas


In [ ]:
#Obter uma lista das variáveis contínuas
continuous_columns = (
    dicionario
    .query('Tipo == "Contínua"')
    .variavel
    .to_list()
)

In [ ]:
if continuous_columns != []:
    for column in df.columns:
        if (column in continuous_columns):
            plt.switch_backend('TkAgg')
            # Gerando gráfico de valores mínimo e máximo
            ## Configurando parâmetros
            ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
            valores = [df[column].min(),df[column].max()]

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Adicionando valores sobre as barras
            ax.bar_label(container=ax.containers[0], labels=valores)

            # Adicionando barra com valor da diferença entre o mínimo e máximo
            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axhline(0, color="k", clip_on=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'.png',bbox_inches='tight')
            plt.close()
            

            # Gerando gráfico de dispersão
            ## Configurando parâmetros
            ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            # Corrigindo valores do eixo y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de dispersão da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-dispersao.png',bbox_inches='tight')        
            plt.close()

            # Gerando gráfico de caixa
            ## Configurando parâmetros
            ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de caixa da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-caixa.png',bbox_inches='tight')
            plt.close()

            # Gráfico de caixa quartiles
            ## Calculando cada quartil da coluna
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            ## Configurando parâmetros
            ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            plt.title('Gráfico de caixa quartil da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'caixa-quartil.png',bbox_inches='tight')        
            plt.close()

## Características discretas

In [ ]:
#Obter uma lista das variáveis discretas
discrete_columns = (
    dicionario
    .query('Tipo == "Discreta"')
    .variavel
    .to_list()
)

In [ ]:
if discrete_columns != []:
    for column in df.columns:
        if (column in discrete_columns):
            # Tabela estatística
            stats = pd.DataFrame({
                'variavel': [column],
                'min': [df[column].min()],
                'max': [df[column].max()],
                'media': [round(df[column].mean(), 2)],
                'mediana': [df[column].median()],
                'std': [round(df[column].std(), 2)],
                'q1': [df[column].quantile(0.25)],
                'q3': [df[column].quantile(0.75)],
            })

            PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
            stats.to_csv(PASTA_TABELAS / f'univariada-discreta-{column}.csv', index=False, encoding='utf-8', sep=';')
            print(f"Tabela salva: univariada-discreta-{column}.csv")

            # 1. Gráfico de valores mínimo e máximo
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.set_style("whitegrid")
            sns.barplot(x=['Mínimo','Máximo'], y=[df[column].min(), df[column].max()], color='#183EFF', alpha=1, ax=ax)

            for bars in ax.containers:
                ax.bar_label(bars)

            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False, ls='dashed')
            sns.despine(left=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica ' + column)

            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'.png', bbox_inches='tight')
            plt.close(fig)

            # 2. Gráfico de dispersão
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.set_style("whitegrid")
            sns.scatterplot(data=df, x=column, y=column, color='#183EFF', ax=ax)
            plt.title('Gráfico de dispersão da característica ' + column)
            plt.xlabel(column)

            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-dispersao.png', bbox_inches='tight')
            plt.close(fig)

            # 3. Gráfico de caixa (Boxplot)
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.set_style("whitegrid")
            sns.boxplot(data=df, x=column, color='#183EFF', medianprops={"color": "#FFD000"}, ax=ax)
            plt.title('Gráfico de caixa da característica ' + column)

            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa.png', bbox_inches='tight')
            plt.close(fig)

            # 4. Gráfico de caixa quartis
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            fig, ax = plt.subplots(figsize=(10, 8))
            sns.set_style("whitegrid")
            sns.boxplot(x=[df[column].min(), q1, q2, q3, df[column].max()], color='#183EFF', medianprops={"color": "#FFD000"}, ax=ax)
            plt.title('Gráfico de caixa quartil da característica ' + column)
            plt.xlabel(column)

            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa-quartil.png', bbox_inches='tight')
            plt.close(fig)


## Características temporais


In [ ]:
#Obter uma lista das variáveis de tempo
data_columns = (
    dicionario
    .query('Tipo == "Temporal"')
    .variavel
    .to_list()
)

In [ ]:
if data_columns != []:
    for column in df.columns:
        if (column in data_columns):

            # Contabilizando quantos registros no ano
            day_data_freq = df[column].value_counts().reset_index()
            day_data_freq.columns = ['data','qnt']

            # Convertendo object em data
            day_data_freq['data'] = pd.to_datetime(day_data_freq['data'], format='%Y') 

            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            # ax.figure.set_size_inches(25, 8)
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            sns.set_style("whitegrid")
            
            ax = sns.lineplot(data=day_data_freq,x='data',y='qnt', palette="tab10", linewidth=2.5, color='#183EFF',alpha=.8)

            ## Adicionando barra com valor médio
            ax.axhline(day_data_freq['qnt'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(column)
            plt.title('Distribuição na linha do tempo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-temporal-'+column+'.png',bbox_inches='tight')
            plt.close()

## Características ordinais

In [ ]:
ordinal_columns = (
    dicionario
    .query('Tipo == "Ordinal"')
    .variavel
    .to_list()
)

In [ ]:
if ordinal_columns != []:
    for column in ordinal_columns:
        # Carregar escala da coluna (DataFrame com coluna/nivel/ordem)
        escala_df_col = escalas_df[escalas_df['coluna'] == column].sort_values('ordem').reset_index(drop=True)
        escala_map = escala_df_col.set_index('nivel')['ordem'].to_dict()

        # Determinar o tipo das categorias na coluna (int ou str)
        cat_dtype = df[column].cat.categories.dtype

        # Contagem por categoria na ordem da escala
        contagens = df[column].value_counts().to_dict()
        rows = []
        for _, row in escala_df_col.iterrows():
            # Converter nivel para o tipo correto da coluna
            nivel_convertido = cat_dtype.type(row['nivel']) if cat_dtype != object else str(row['nivel'])
            rows.append({
                'categoria': escala_map.get(nivel_convertido, str(row['nivel'])),
                'nivel': nivel_convertido,
                'ordem': int(row['ordem']),
                'frequencia': contagens.get(nivel_convertido, 0),
            })
        freq_df = pd.DataFrame(rows).sort_values('ordem').reset_index(drop=True)
        freq_df['percentual'] = (freq_df['frequencia'] / len(df) * 100).round(2)

        # Tabela salva
        PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
        tabela_path = PASTA_TABELAS / (f'univariada-ordinal-' + column + '.csv')
        freq_df.to_csv(tabela_path, index=False, encoding='utf-8', sep=';')
        print(f"Tabela salva: {tabela_path}")

        # Gráfico de barras com frequência
        sns.set_style("whitegrid")
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao)

        ax = sns.barplot(data=freq_df, x='categoria', y='frequencia', color='#183EFF', alpha=1, ax=ax)
        for bars in ax.containers:
            ax.bar_label(bars)

        plt.ylabel('Frequência')
        plt.xlabel(column)
        plt.title(f'Distribuição de frequências — {column}')
        plt.xticks(rotation=30, ha='right')

        fig.savefig('../docs/imagens/eda/univariada-ordinal-'+column+'-frequencia.png', bbox_inches='tight')
        plt.close(fig)

        # Gráfico de percentual
        fig, ax = plt.subplots(figsize=padrao)
        ax = sns.barplot(data=freq_df, x='categoria', y='percentual', color='#183EFF', alpha=1, ax=ax)
        for bars in ax.containers:
            ax.bar_label(bars, fmt='%.1f%%')

        plt.ylabel('Percentual (%)')
        plt.xlabel(column)
        plt.title(f'Distribuição percentual — {column}')
        plt.xticks(rotation=30, ha='right')

        fig.savefig('../docs/imagens/eda/univariada-ordinal-'+column+'-percentual.png', bbox_inches='tight')
        plt.close(fig)


# Análise Bivariada

## Nominal X Contínuas

In [ ]:
if (continuous_columns != []) and (nominal_columns != []):
    for column1 in nominal_columns:
        for column2 in continuous_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada = bivariada.sort_values(by=column2, ascending=False).reset_index(drop=True)

            # Salvar tabela
            PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
            bivariada.to_csv(PASTA_TABELAS / f'bivariada-{column2}_X_{column1}.csv', index=False, encoding='utf-8', sep=';')
            print(f"Tabela salva: bivariada-{column2}_X_{column1}.csv")

            fig, ax = plt.subplots(figsize=(25, 8))
            sns.set_style("whitegrid")

            sns.barplot(data=bivariada.head(10), x=column2, y=column1, color='#183EFF', alpha=1, ax=ax)

            for bars in ax.containers:
                ax.bar_label(bars)

            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False, ls='dashed')

            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise bivariada — média de ' + column2 + ' por ' + column1)

            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png', bbox_inches='tight')
            plt.close(fig)


## Nominal x Discretas

In [ ]:
if (nominal_columns != []) and (discrete_columns != []):
    for column1 in nominal_columns:
        for column2 in discrete_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada = bivariada.sort_values(by=column2, ascending=False).reset_index(drop=True)

            # Salvar tabela
            PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
            bivariada.to_csv(PASTA_TABELAS / f'bivariada-{column2}_X_{column1}.csv', index=False, encoding='utf-8', sep=';')
            print(f"Tabela salva: bivariada-{column2}_X_{column1}.csv")

            fig, ax = plt.subplots(figsize=(25, 8))
            sns.set_style("whitegrid")

            sns.barplot(data=bivariada.head(10), x=column2, y=column1, color='#183EFF', alpha=1, ax=ax)

            for bars in ax.containers:
                ax.bar_label(bars)

            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False, ls='dashed')

            current_values = ax.get_xticks()
            ax.set_xticks(current_values)
            ax.set_xticklabels([f'{x:.0f}' for x in current_values])

            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise bivariada — média de ' + column2 + ' por ' + column1)

            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png', bbox_inches='tight')
            plt.close(fig)


## Nominal x Tempo

In [ ]:
pallette = [
'#00F537', 
'#FFFF00',
'#00C8FF',
'#FF6464',
'#00D000',
'#FFD000',
'#183EFF',
'#3C3C3C',
'#FF0000',
'#009600',
'#FF7D00',
'#00008C'
]

In [ ]:
data_columns = (
    dicionario
    .query('Tipo == "Temporal"')
    .variavel
    .to_list()
)

In [ ]:
if (nominal_columns != []) and (data_columns != []):
    for time in data_columns:
        for nominal in nominal_columns:
            # Identificando as categorias mais frequentes
            grouped = df.groupby([nominal])[nominal].count().reset_index(name='Contagem').sort_values(by='Contagem',ascending=False)[:12]
            grouped = grouped[nominal].tolist()
            
            # Agrupando as categorias pelo tempo
            grouped_category = df.groupby([time,nominal])[nominal].count().reset_index(name='Contagem')
            
            # Selecionando as categorias mais frequentes
            grouped_category = grouped_category[grouped_category[nominal].isin(grouped)]

            # Convertendo coluna do tempo em datetime
            grouped_category[time] = pd.to_datetime(grouped_category[time], format='%Y')
            
            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            sns.set_style("whitegrid")
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            
            ax = sns.lineplot(data=grouped_category,
                              x=time,
                              y='Contagem', 
                              hue=nominal,
                              palette=pallette[0:len(grouped)], 
                              linewidth=2.5, 
                              color='#183EFF',
                              alpha=1)

            ## Adicionando barra com valor médio
            ax.axhline(grouped_category['Contagem'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(time)
            plt.title('Distribuição na linha do tempo da característica '+ time + ' x ' + nominal)
            
            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+nominal+' X '+time+'.png',bbox_inches='tight')   
            plt.close()

## Nominal x Ordinal

In [ ]:
# True = percentual por linha (compara a distribuição entre categorias nominais)
# False = contagem absoluta
normalizar = True

# Garantindo que as variáveis são listas, e não strings
if isinstance(nominal_columns, str):
    nominal_columns = [nominal_columns]
if isinstance(ordinal_columns, str):
    ordinal_columns = [ordinal_columns]

if (nominal_columns != []) and (ordinal_columns != []):
    for ordinal in ordinal_columns:
        # A ordem dos níveis vem da escala aplicada no carregamento (aplicar_escalas)
        if not (isinstance(df[ordinal].dtype, pd.CategoricalDtype) and df[ordinal].cat.ordered):
            print(f'Aviso: "{ordinal}" não tem escala aplicada; coluna ignorada. '
                  f'Confira o escala.csv e rode aplicar_escalas antes deste script.')
            continue
        ordem = list(df[ordinal].cat.categories)

        for nominal in nominal_columns:
            if nominal == ordinal:
                continue

            # Identificando as 12 categorias nominais mais frequentes
            top = df[nominal].value_counts().head(12).index.tolist()
            subset = df[df[nominal].isin(top)]

            # Tabela de contingência (linhas = nominal, colunas = ordinal na ordem da escala)
            tabela = pd.crosstab(subset[nominal], subset[ordinal])
            tabela = tabela.reindex(index=top, columns=ordem, fill_value=0)

            if normalizar:
                dados = tabela.div(tabela.sum(axis=1), axis=0) * 100
                fmt = '.1f'
                legenda = '% dos registros da categoria'
            else:
                dados = tabela
                fmt = 'd'
                legenda = 'Quantidade de registros'

            # Gerando gráfico
            sns.set_style("white")
            largura = max(8, 1.2 * len(ordem) + 4)
            altura = max(5, 0.6 * len(top) + 2)
            fig, ax = plt.subplots(figsize=(largura, altura))

            sns.heatmap(dados,
                        annot=True,
                        fmt=fmt,
                        cmap='Blues',
                        linewidths=0.5,
                        linecolor='white',
                        cbar_kws={'label': legenda},
                        ax=ax)

            ax.set_xticklabels([textwrap.fill(str(n), 14) for n in ordem], rotation=0)
            plt.ylabel(nominal)
            plt.xlabel(ordinal)
            plt.yticks(rotation=0)
            plt.title('Tabela de contingência da característica ' + nominal + ' x ' + ordinal)

            # Salvando figura
            fig.savefig('../docs/imagens/eda/bivariada-' + nominal + ' X ' + ordinal + '.png',
                        bbox_inches='tight')
            plt.close(fig)

            # Salvar tabela de contingência
            tabela_salva = dados.copy()
            tabela_salva.index.name = nominal
            PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
            tabela_salva.to_csv(PASTA_TABELAS / (f'bivariada-' + nominal + ' X ' + ordinal + '.csv'), sep=';')
            print(f"Tabela salva: bivariada-{nominal}_X_{ordinal}.csv")


# Análise Multivariada

In [ ]:
# Análise Multivariada: Correlograma das variáveis discretas
fig, ax = plt.subplots(figsize=(22, 18))
sns.set_style("whitegrid")

discrete_columns = dicionario.query('Tipo == "Discreta"').variavel.to_list()
df_numerico = df[discrete_columns]

matriz_correlacao = df_numerico.corr()

sns.heatmap(matriz_correlacao, annot=False, cmap='coolwarm', linewidths=0.5, ax=ax)

plt.title('Correlograma das Características Discretas (RH)', fontsize=16)

# Salvar figura
fig.savefig('../docs/imagens/eda/multivariada-correlograma.png', bbox_inches='tight')
plt.close(fig)

# Salvar tabela da matriz de correlação
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
matriz_correlacao.to_csv(PASTA_TABELAS / 'multivariada-correlograma.csv', sep=';')
print("Tabela salva: multivariada-correlograma.csv")


# Análise de valores ausentes

In [ ]:
# Contabilizando valores ausentes por variável
df_original = pd.read_csv('../data/raw/base_rh_ptBR.csv')
counts = df_original.isna().sum().reset_index()
counts.columns = ['variavel', 'qnt_ausentes']
counts['percentual'] = (counts['qnt_ausentes'] / len(df_original) * 100).round(2)
counts = counts.sort_values('qnt_ausentes', ascending=False).reset_index(drop=True)


In [ ]:
## Configurando estilo e tamanho
sns.set_style("whitegrid")
padrao = (25, 12)
fig, ax = plt.subplots(figsize=padrao)

## Gerando gráfico de barras com a quantidade de ausências por variável
ax = sns.barplot(data=counts, x='variavel', y='qnt_ausentes', color='#183EFF', alpha=1, ax=ax)

# Personalizando valores absolutos
for i in ax.containers:
    ax.bar_label(i)

## Configurações de cores e barras
sns.despine(left=False)
ax.axhline(0, color="k", clip_on=False)
ax.tick_params(axis='x', rotation=45)
plt.xlabel('Características')
plt.ylabel('Quantidade de ausências')
plt.title('Análise do quantitativo de valores ausentes')

## Salvando figura
fig.savefig('../docs/imagens/eda/valores-ausentes.png', bbox_inches='tight', dpi=300)
plt.close(fig)

## Salvando tabela de valores ausentes
PASTA_TABELAS = Path('../docs/tabelas')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
counts.to_csv(PASTA_TABELAS / 'valores-ausentes.csv', index=False, encoding='utf-8', sep=';')


# Detecção de valores discrepantes

In [ ]:
# Criando função para calcular os limites de outlier pelo método IQR
def limites_IQR(data):
    """Retorna os limites abaixo/acima dos quais um valor é considerado outlier."""
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR


# Mapeando as colunas
outliers_columns = discrete_columns + continuous_columns

# Estrutura de repetição para mapear os outliers de cada variável
linhas = []
for column in outliers_columns:
    dados = df[column].dropna()
    limite_inferior, limite_superior = limites_IQR(dados)

    inferiores = dados[dados < limite_inferior]
    superiores = dados[dados > limite_superior]

    # Variáveis sem nenhum outlier ficam fora da tabela
    if inferiores.empty and superiores.empty:
        continue

    linhas.append({
        'variavel': column,
        'ponta_inferior': inferiores.min() if not inferiores.empty else None,
        'ponta_inferior_inicio': round(limite_inferior, 2) if not inferiores.empty else None,
        'ponta_superior_inicio': round(limite_superior, 2) if not superiores.empty else None,
        'ponta_superior': superiores.max() if not superiores.empty else None,
    })

outliers = pd.DataFrame(linhas)

# Renomeando as colunas para a leitura da área de negócio
outliers = outliers.rename(columns={
    'variavel': 'Característica',
    'ponta_inferior': 'Menor valor atípico',
    'ponta_inferior_inicio': 'Atípico abaixo de',
    'ponta_superior_inicio': 'Atípico acima de',
    'ponta_superior': 'Maior valor atípico',
})

# Salvando outliers
Path('../docs/tabelas').mkdir(parents=True, exist_ok=True)
outliers.to_csv('../docs/tabelas/outliers.csv', sep=';', index=False)
outliers

# Correlação de variáveis

In [ ]:
def correlograma_circulos(corr, titulo='Matriz de correlação', anotar=None,
                         cmap=None, figsize=None, ax=None):
    """Correlograma com círculos no triângulo inferior.
 
    A área de cada círculo é proporcional à força da correlação (|r|):
    círculo grande = forte, pequeno = fraca. A cor mostra o sinal e o valor,
    de -1 (azul) a 1 (vermelho).
    """
    n = len(corr)
    if anotar is None:
        anotar = n <= 15  # com muitas variáveis os números ficam ilegíveis
    if cmap is None:
        cmap = sns.diverging_palette(230, 20, as_cmap=True)
    if figsize is None:
        lado = max(6, 0.45 * n + 2)
        figsize = (lado + 1.5, lado)
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
 
    valores = corr.to_numpy()
 
    linhas, colunas = np.tril_indices(n, k=-1)
    r = valores[linhas, colunas]
 
    x = colunas
    y = linhas - 1
 
    diametros = 0.9 * np.sqrt(np.abs(r))
 
    circulos = EllipseCollection(
        widths=diametros,
        heights=diametros,
        angles=np.zeros_like(r, dtype=float),
        units='x',
        offsets=np.column_stack([x, y]),
        offset_transform=ax.transData,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.3,
    )
    circulos.set_array(r)
    circulos.set_clim(-1, 1)
    ax.add_collection(circulos)
 
    if anotar:
        for xi, yi, ri in zip(x, y, r):
            ax.text(xi, yi, f'{ri:.2f}'.replace('0.', '.', 1) if abs(ri) < 1 else f'{ri:.0f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(ri) > 0.6 else '#333333')  # círculos pequenos: texto escuro
 
    # Eixos e grade
    nomes = list(corr.columns)
    ax.set_xticks(range(n - 1))
    ax.set_xticklabels(nomes[:-1], rotation=90)
    ax.set_yticks(range(n - 1))
    ax.set_yticklabels(nomes[1:])
    ax.set_xlim(-0.5, n - 1.5)
    ax.set_ylim(n - 1.5, -0.5)
    ax.set_aspect('equal')
    ax.set_xticks(np.arange(-0.5, n - 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n - 1), minor=True)
    ax.grid(which='minor', color='#EEEEEE', linewidth=0.6)
    ax.tick_params(which='minor', length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)
 
    barra = fig.colorbar(circulos, ax=ax, shrink=0.5, pad=0.02)
    barra.set_label('Correlação')
    barra.outline.set_visible(False)
 
    ax.set_title(titulo)
    return fig, ax

# Seleção de características

In [ ]:
# Mantendo apenas as colunas que serão utilizadas no modelo
df_model = df.drop(columns=['Numero_do_Funcionario', 'Contagem_de_Funcionarios', 'Horas_Padrao', 'Maior_de_18'])

# Salvando base final processado para a fase de modelagem
df_model.to_csv('../data/processed/base_rh_columns_selected.csv', index=False, sep=';')

# Insights

1. **Estrutura e Integridade dos Dados:**
   - A base de dados é composta por 1.470 registros e 35 colunas originais, apresentando **0% de valores nulos (ausentes)**, o que garantiu uma análise limpa sem necessidade de imputação de dados.
   - Foram identificadas e removidas colunas irrelevantes para a modelagem, como IDs unívocos (`Numero_do_Funcionario`) e colunas com desvio padrão zero (`Contagem_de_Funcionarios`, `Horas_Padrao`, `Maior_de_18`).

2. **Distribuição Demográfica e Organizacional:**
   - A idade dos colaboradores varia majoritariamente entre **18 e 60 anos**, com média em torno de 37 anos, refletindo um quadro profissional maduro.
   - As análises univariadas e bivariadas permitiram mapear a distribuição de funcionários por **Departamento**, **Cargo**, **Gênero** e **Estado Civil**, evidenciando quais áreas concentram maior volume de pessoal.

3. **Correlações e Padrões Numéricos:**
   - Através do correlograma gerado com as variáveis discretas, foi possível identificar fortes correlações lógicas de negócio (por exemplo, a relação natural entre o avanço da idade, tempo de carreira e a progressão da renda mensal).
   - A ausência de variáveis estritamente contínuas ou temporais direcionou o foco da exploração para as métricas discretas, nominais e ordinais.

4. **Variável-Alvo e Segmentação:**
   - A variável **`Desligamento`** (Sim/Não) possui distribuição desbalanceada: a grande maioria dos registros corresponde a colaboradores ativos. Em um projeto de segmentação não supervisionada, recomenda-se **não incluir `Desligamento` na formação dos clusters**, pois ela representa um resultado de interesse e não um traço descritivo. O papel ideal é usá-la posteriormente para **caracterizar e avaliar** os clusters formados — por exemplo, verificando se algum grupo apresenta taxa de desligamento significativamente maior.

5. **Limitações e Próximos Passos:**
   - A base exportada em `base_rh_columns_selected.csv` **não está ainda pronta para modelagem**. Restam etapas de curadoria nos notebooks 02 e 03, que incluem:
     - Codificação das variáveis nominais (one-hot / target encoding) e manutenção da ordem das ordinais;
     - Tratamento de outliers identificados na detecção discrepante;
     - Normalização ou padronização das variáveis numéricas;
     - Seleção e avaliação de features antes do ajuste do modelo.
   - Apenas após a conclusão dessas etapas a base estará apta para a aplicação de algoritmos de segmentação (ex: K-Means, DBSCAN) ou classificação supervisionada.
